# Vi-PPE V6 Judge Experiment

V6 tách biệt với V5. Notebook này có `MODELS` selector ở đầu: để rỗng thì chạy cả 4 model; điền danh sách key thì chỉ chạy model được chọn.


## 1. Setup

In [ ]:
from pathlib import Path
import json

ROOT = Path('/content/Vi-PPE-mini')
if ROOT.exists():
    %cd /content/Vi-PPE-mini
else:
    print('Repo path /content/Vi-PPE-mini not found. Clone or mount it before continuing.')

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes

In [ ]:
!python - <<'PY'
import torch
print('cuda_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print('total_gb:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
PY
!git rev-parse --short HEAD || true

## 2. Select Models

Để `MODELS = []` hoặc `MODELS = None` thì notebook chạy đủ 4 model. Muốn chạy một phần, đặt ví dụ `MODELS = ["qwen25_7b", "llama31_8b_instruct"]`.

In [ ]:
AVAILABLE_MODELS = {
    "qwen25_3b": {
        "model": "qwen25_3b_a100_40gb_b64",
        "run_prefix": "qwen25_3b",
        "run_suffix": "a100_b64",
        "note": "Qwen2.5-3B-Instruct bf16 batch_size 64",
    },
    "qwen25_7b": {
        "model": "qwen25_7b_a100_40gb_b16",
        "run_prefix": "qwen25_7b",
        "run_suffix": "a100_b16",
        "note": "Qwen2.5-7B-Instruct bf16 batch_size 16",
    },
    "gemma3_4b_it": {
        "model": "gemma3_4b_it_a100_40gb_b16",
        "run_prefix": "gemma3_4b_it",
        "run_suffix": "a100_b16",
        "note": "Gemma-3-4B-IT bf16 batch_size 16",
    },
    "llama31_8b_instruct": {
        "model": "llama31_8b_instruct_a100_40gb_b16",
        "run_prefix": "llama31_8b_instruct",
        "run_suffix": "a100_b16",
        "note": "Llama-3.1-8B-Instruct bf16 batch_size 16; requires HF_TOKEN and accepted Meta license",
    },
}

# Edit this list when you want a subset. Empty/None means run all 4 models.
MODELS = []
# MODELS = ["qwen25_7b"]
# MODELS = ["qwen25_3b", "qwen25_7b"]
# MODELS = ["gemma3_4b_it", "llama31_8b_instruct"]

selected_keys = list(AVAILABLE_MODELS) if not MODELS else list(MODELS)
unknown = sorted(set(selected_keys) - set(AVAILABLE_MODELS))
if unknown:
    raise ValueError(f"Unknown MODELS entries: {unknown}. Available: {list(AVAILABLE_MODELS)}")

SELECTED_MODELS = {key: AVAILABLE_MODELS[key] for key in selected_keys}
print("Selected models:")
for key, cfg in SELECTED_MODELS.items():
    print(f"- {key}: {cfg['model']} ({cfg['note']})")

## 3. Dataset, Prompt, And Config Check

In [ ]:
for path in [
    'data/processed/pairs_test_llm_v4.jsonl',
    'data/processed/bias_subset_llm_v4.jsonl',
    'configs/v6_runs.yaml',
    'prompts/baseline_generic_v6_vi.md',
    'prompts/criteria_instruction_strict_v6_vi.md',
    'prompts/criteria_bias_mitigated_v6_vi.md',
]:
    p = Path(path)
    print(path, 'OK' if p.exists() else 'MISSING')

In [ ]:
!python -m pytest tests/test_prompt_render.py tests/test_bias_metrics.py -q

In [ ]:
!python src/vi_ppe/prompt_render.py \
  --dataset data/processed/pairs_test_llm_v4.jsonl \
  --pair-id llm_core_instruction_000112 \
  --template auto_criteria_v6_by_domain \
  --order AB | head -120

## 4. Smoke Runs

In [ ]:
# Smoke criteria V6 cho các model được chọn, limit 20 để kiểm tra parser/prompt trước.
for key, cfg in SELECTED_MODELS.items():
    run_id = f"{cfg['run_prefix']}_criteria_test_llm_v6_{cfg['run_suffix']}_smoke"
    model = cfg['model']
    print(f"\n=== Smoke: {key} -> {run_id} ===")
    !python scripts/05_run_judge.py       --dataset data/processed/pairs_test_llm_v4.jsonl       --model {model}       --template auto_criteria_v6_by_domain       --run-id {run_id}       --limit 20       --resume
    !python scripts/06_compute_metrics.py       --judgments results/judgments/{run_id}.jsonl       --dataset data/processed/pairs_test_llm_v4.jsonl       --run-id {run_id}       --limit 20

## 5. Main V6 Runs

In [ ]:
V6_EXPERIMENTS = [
    {
        "suffix": "baseline_test_llm_v6",
        "dataset": "data/processed/pairs_test_llm_v4.jsonl",
        "template": "baseline_generic_v6_vi",
        "bias": False,
    },
    {
        "suffix": "criteria_test_llm_v6",
        "dataset": "data/processed/pairs_test_llm_v4.jsonl",
        "template": "auto_criteria_v6_by_domain",
        "bias": False,
    },
    {
        "suffix": "bias_mitigated_llm_v6",
        "dataset": "data/processed/bias_subset_llm_v4.jsonl",
        "template": "criteria_bias_mitigated_v6_vi",
        "bias": True,
    },
    {
        "suffix": "bias_instruction_strict_llm_v6",
        "dataset": "data/processed/bias_subset_llm_v4.jsonl",
        "template": "criteria_bias_instruction_strict_v6_vi",
        "bias": True,
    },
]

RUN_IDS = []
for key, cfg in SELECTED_MODELS.items():
    for exp in V6_EXPERIMENTS:
        run_id = f"{cfg['run_prefix']}_{exp['suffix']}_{cfg['run_suffix']}"
        RUN_IDS.append(run_id)
        model = cfg['model']
        dataset = exp['dataset']
        template = exp['template']
        bias_flag = "--bias" if exp['bias'] else ""
        print(f"\n=== Running {run_id} ===")
        print(f"model={model} dataset={dataset} template={template}")
        !python scripts/05_run_judge.py           --dataset {dataset}           --model {model}           --template {template}           --run-id {run_id}           --resume
        !python scripts/06_compute_metrics.py           --judgments results/judgments/{run_id}.jsonl           --dataset {dataset}           --run-id {run_id}           {bias_flag}

## 6. Optional Single-Model Example

In [ ]:
# Selector ở đầu đã xử lý Llama/Gemma/Qwen.
# Ví dụ chỉ chạy Llama: đặt MODELS = ["llama31_8b_instruct"] rồi chạy lại các cell từ phần Main V6 Runs.
# Llama cần HF_TOKEN và bạn phải accept Meta license trên Hugging Face.

## 7. Summary Loader

In [ ]:
summary_ids = globals().get('RUN_IDS')
if not summary_ids:
    summary_ids = []
    for key, cfg in SELECTED_MODELS.items():
        for suffix in [
            'baseline_test_llm_v6',
            'criteria_test_llm_v6',
            'bias_mitigated_llm_v6',
            'bias_instruction_strict_llm_v6',
        ]:
            summary_ids.append(f"{cfg['run_prefix']}_{suffix}_{cfg['run_suffix']}")

for run_id in summary_ids:
    p = Path('results/metrics') / f'{run_id}_summary.json'
    if not p.exists():
        print('\n===', run_id, 'MISSING ===')
        continue
    data = json.loads(p.read_text(encoding='utf-8'))
    print('\n===', run_id, '===')
    for key in ['num_pairs', 'parse_success_rate', 'pairwise_accuracy', 'macro_accuracy', 'lower_bound_domain_score', 'swap_consistency', 'invalid_count', 'inconsistent_count']:
        print(f'{key}:', data.get(key))
    print('domain_accuracy:', data.get('domain_accuracy'))
    bias_path = Path('results/metrics') / f'{run_id}_bias_summary.json'
    if bias_path.exists():
        bias = json.loads(bias_path.read_text(encoding='utf-8'))
        print('verbosity_bias_rate:', bias.get('verbosity_bias_rate'))
        print('style_bias_rate:', bias.get('style_bias_rate'))
        print('style_bias_coverage:', bias.get('style_bias_coverage'))